# Privacy Leakage Evaluation — Detective-Game Benchmark
**CS 466 | Yilin Pan & Tyler Hudgins**

Compares three preprocessing modes on identity-matching accuracy and sensitive attribute recovery:

| Mode | Description | Expected leakage |
|------|-------------|------------------|
| `raw` | No preprocessing — original images sent directly | Highest |
| `presidio_only` | Text PII redacted; images unmasked | Intermediate |
| `our_filter` | Full pipeline: text redaction + visual masking | Lowest |

The downstream LLM agent is the same (GPT-4o) in all conditions — only preprocessing changes.

---
## Section 0 — Configuration

In [ ]:
# ── Experiment switches ────────────────────────────────────────────────────────
QUICK_TEST   = True   # True → one case, raw mode only, 3 images. Set False for full eval.
RANDOM_SEED  = 42     # Reproducible image-label shuffling
TEMPERATURE  = 0      # Deterministic agent responses

# Modes to run in full evaluation (order preserved in tables)
EVAL_MODES   = ["raw", "presidio_only", "our_filter"]

# ── Paths ─────────────────────────────────────────────────────────────────────
import os, sys
from pathlib import Path

PROJECT_ROOT   = Path(os.path.abspath(""))
DATASET_ROOT   = PROJECT_ROOT / "demo_dataset" / "candidates_test"
CONFIG_PATH    = PROJECT_ROOT / "config" / "test_inputs.yaml"
PROFILES_PATH  = DATASET_ROOT / "candidates_profile.json"
GAME_PATH      = DATASET_ROOT / "game_prompt.json"
OUTPUTS_DIR    = PROJECT_ROOT / "outputs" / "evaluation"
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root : {PROJECT_ROOT}")
print(f"Quick test   : {QUICK_TEST}")
print(f"Modes        : {EVAL_MODES if not QUICK_TEST else ['raw']}")

---
## Section 1 — Imports & Helpers

In [ ]:
import base64
import io
import json
import random
import re
import time
from copy import deepcopy
from difflib import SequenceMatcher

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml
from dotenv import load_dotenv
from openai import OpenAI
from PIL import Image

load_dotenv(override=True)
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "")
if not OPENAI_API_KEY:
    raise RuntimeError("OPENAI_API_KEY not found. Add it to .env or set the env var.")

client = OpenAI(api_key=OPENAI_API_KEY)
rng    = random.Random(RANDOM_SEED)
print("Imports OK. OpenAI client ready.")


# ── Image → base64 ────────────────────────────────────────────────────────────
def pil_to_b64(img: Image.Image, fmt: str = "PNG") -> str:
    buf = io.BytesIO()
    img.save(buf, format=fmt)
    return base64.b64encode(buf.getvalue()).decode()


# ── Robust JSON extraction from LLM output ────────────────────────────────────
def extract_json(text: str) -> dict:
    """Extract the first valid JSON object from an LLM response string."""
    text = text.strip()
    # Strip markdown code fences
    text = re.sub(r"^```(?:json)?\s*", "", text, flags=re.MULTILINE)
    text = re.sub(r"```\s*$", "", text, flags=re.MULTILINE)
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        # Find the first {...} block
        m = re.search(r"\{.*\}", text, re.DOTALL)
        if m:
            return json.loads(m.group())
        return {}


# ── Fuzzy string match ────────────────────────────────────────────────────────
def fuzzy_match(pred, truth, threshold: float = 0.65) -> bool:
    if pred is None or truth is None:
        return False
    a, b = str(pred).strip().lower(), str(truth).strip().lower()
    return a == b or SequenceMatcher(None, a, b).ratio() >= threshold


# Exact match for structured fields (IDs, card numbers, codes)
EXACT_FIELDS = {"ID Number", "Bank Card Number", "Flight Number", "License Plate Number"}

def field_match(field: str, pred, truth) -> bool:
    if pred is None or truth is None:
        return False
    if field in EXACT_FIELDS:
        return str(pred).strip() == str(truth).strip()
    return fuzzy_match(pred, truth)

print("Helper functions defined.")

---
## Section 2 — Load Dataset

In [ ]:
# ── Prompt templates from YAML ─────────────────────────────────────────────────
with open(CONFIG_PATH) as f:
    cfg = yaml.safe_load(f)

DET_CFG         = cfg["detective_game"]
SYSTEM_PROMPT   = DET_CFG["system_prompt"].strip()
USER_TEMPLATE   = DET_CFG["user_prompt_template"]

# ── Candidate profiles (detective-style, no raw PII) ──────────────────────────
with open(PROFILES_PATH) as f:
    PROFILES = json.load(f)   # {candidate_A: {true_person_id, profile}, ...}

# ── Game cases (evidence file lists + ground truth) ───────────────────────────
with open(GAME_PATH) as f:
    GAME_CFG = json.load(f)
CASES = GAME_CFG["cases"]

# ── Ground truth sensitive attributes (for metric calculation only) ───────────
GROUND_TRUTH: dict[str, dict] = {}
for case in CASES:
    pid = case["true_person_id"]
    gt_path = DATASET_ROOT / pid / f"{pid}.json"
    with open(gt_path) as f:
        data = json.load(f)
    record = data[0] if isinstance(data, list) else data
    GROUND_TRUTH[pid] = {
        "Name":               record.get("Name"),
        "ID Number":          record.get("ID Number"),
        "Address":            record.get("Address"),
        "Bank Card Number":   record.get("Bank Card Number"),
        "Flight Number":      record.get("Flight Number"),
        "License Plate Number": record.get("License Plate Number"),
        "Diagnosis":          record.get("Diagnosis"),
        "Salary":             record.get("Job Position Salary"),
        "Relationship":       record.get("Relationship in the Same Frame"),
    }

print(f"Loaded {len(CASES)} cases, {len(PROFILES)} candidate profiles")
print(f"Ground-truth persons: {list(GROUND_TRUTH.keys())}")
for cid, meta in PROFILES.items():
    print(f"  {cid} → {meta['true_person_id']}")

---
## Section 3 — Privacy Pipeline (for `presidio_only` and `our_filter` modes)

Skip this cell for quick test — models are only needed for the two filtered modes.

In [ ]:
_models_loaded = False

if not QUICK_TEST and ("presidio_only" in EVAL_MODES or "our_filter" in EVAL_MODES):
    from src.privacy_pipeline import (
        initialize_models, get_models,
        detect_privacy_risks_from_image, privacy_gate_and_embed,
    )
    initialize_models(load_inpainter=False)   # skip Stable Diffusion for speed
    _pipeline_models = get_models()
    _models_loaded = True
    print("Privacy pipeline models loaded.")
else:
    print("Quick-test mode — skipping model load (raw mode only).")

---
## Section 4 — Preprocessing Functions

In [ ]:
def preprocess_raw(images: list[Image.Image]) -> list[Image.Image | None]:
    """No preprocessing — return originals."""
    return list(images)


def preprocess_presidio_only(images: list[Image.Image]) -> list[Image.Image | None]:
    """Text PII redacted (via OCR+NER) but images passed unmasked.

    The key demonstration: the agent still sees all visual content.
    OCR-extracted text strings are redacted, but the image is unchanged.
    GPT-4o will still read any visible text from the raw image directly.
    """
    # Images are returned unchanged — text redaction only applies to
    # any separately provided text context, not the image pixels.
    return list(images)


def preprocess_our_filter(images: list[Image.Image]) -> list[Image.Image | None]:
    """Full privacy pipeline: detect sensitive regions, apply black-box masking.

    Returns None for images where the entire content is suppressed (ABSTRACT policy).
    """
    if not _models_loaded:
        raise RuntimeError("Run the model-loading cell before using our_filter mode.")

    processed = []
    for img in images:
        boxes = detect_privacy_risks_from_image(img)
        policy, final_img, summary, _, _ = privacy_gate_and_embed(
            image=img, sensitive_boxes=boxes
        )
        print(f"    [{policy}] {len(boxes)} box(es) detected")
        processed.append(final_img)   # None if ABSTRACT
    return processed


PREPROCESSORS = {
    "raw":           preprocess_raw,
    "presidio_only": preprocess_presidio_only,
    "our_filter":    preprocess_our_filter,
}

print("Preprocessing functions defined.")

---
## Section 5 — Agent Call

In [ ]:
def format_candidate_profiles(profiles: dict) -> str:
    """Render all candidate profiles as numbered paragraphs for the prompt."""
    lines = []
    for cid, meta in profiles.items():
        lines.append(f"[{cid.upper()}]  {meta['profile']}")
    return "\n\n".join(lines)


def format_evidence_block(labels: list[str], suppressed: list[bool]) -> str:
    """Describe the evidence items in the prompt text (images are attached separately)."""
    lines = []
    for label, is_suppressed in zip(labels, suppressed):
        if is_suppressed:
            lines.append(f"{label}: [This evidence item was suppressed — high sensitivity detected.]")
        else:
            lines.append(f"{label}: [See attached image]")
    return "\n".join(lines)


def call_detective_agent(
    case_id: str,
    labeled_images: list[tuple[str, Image.Image | None]],
    mode: str,
) -> tuple[dict, str]:
    """Send formatted evidence to GPT-4o and return (parsed_result, raw_text).

    Images labeled None (ABSTRACT policy) are described in text but not attached.
    Neutral labels (Image 1, Image 2 …) are used — never original filenames.
    """
    labels     = [lbl for lbl, _ in labeled_images]
    suppressed = [img is None for _, img in labeled_images]
    active_imgs = [img for _, img in labeled_images if img is not None]

    candidate_profiles_text = format_candidate_profiles(PROFILES)
    evidence_block_text     = format_evidence_block(labels, suppressed)

    # Presidio note added as an in-prompt hint so the agent knows text was filtered
    presidio_note = ""
    if mode == "presidio_only":
        presidio_note = (
            "\n\nNote: Any text evidence associated with this case has had "
            "personally identifiable tokens replaced with [REDACTED] markers. "
            "Visual evidence is provided as-is."
        )

    user_text = USER_TEMPLATE.format(
        case_id=case_id,
        candidate_profiles=candidate_profiles_text,
        evidence_block=evidence_block_text + presidio_note,
    )

    # Build multimodal message: text first, then images
    content: list[dict] = [{"type": "text", "text": user_text}]
    for img in active_imgs:
        b64 = pil_to_b64(img)
        content.append({
            "type": "image_url",
            "image_url": {"url": f"data:image/png;base64,{b64}", "detail": "high"},
        })

    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": content},
        ],
        max_tokens=1200,
        temperature=TEMPERATURE,
    )

    raw_text = response.choices[0].message.content or ""
    parsed   = extract_json(raw_text)
    return parsed, raw_text


print("Agent call functions defined.")

---
## Section 6 — Evaluation Runner

In [ ]:
def run_case(case: dict, mode: str, quick_test: bool = False) -> dict:
    """Run one case under one mode. Returns a result dict."""
    person_dir   = DATASET_ROOT / case["true_person_id"]
    files        = case["quick_test_subset"] if quick_test else case["evidence_files"]

    # Load images
    images = [Image.open(person_dir / f).convert("RGB") for f in files]

    # Apply preprocessing
    print(f"  Preprocessing [{mode}] for {case['case_id']} …")
    processed = PREPROCESSORS[mode](images)

    # Shuffle + assign neutral labels (no filenames revealed to agent)
    indexed = list(enumerate(processed))
    rng.shuffle(indexed)
    labeled_images = [(f"Image {i+1}", processed[orig_idx])
                      for i, (orig_idx, _) in enumerate(indexed)]

    # Call agent
    print(f"  Calling GPT-4o …")
    t0             = time.perf_counter()
    parsed, raw    = call_detective_agent(case["case_id"], labeled_images, mode)
    elapsed        = round(time.perf_counter() - t0, 2)

    # Score identity match
    predicted_cid  = parsed.get("predicted_candidate_id", "unknown")
    is_correct     = (predicted_cid == case["true_candidate_id"])
    confidence     = float(parsed.get("confidence", 0.0))

    # Score attribute recovery vs ground truth
    gt             = GROUND_TRUTH[case["true_person_id"]]
    recovered      = parsed.get("recovered_attributes", {})
    tp = fp = fn   = 0
    for field, gt_val in gt.items():
        pred_val = recovered.get(field)
        if gt_val is None:
            continue
        if field_match(field, pred_val, gt_val):
            tp += 1
        elif pred_val is not None:
            fp += 1
        else:
            fn += 1

    total_gt_fields = sum(1 for v in gt.values() if v is not None)

    return {
        "case_id":              case["case_id"],
        "mode":                 mode,
        "true_person_id":       case["true_person_id"],
        "true_candidate_id":    case["true_candidate_id"],
        "predicted_candidate_id": predicted_cid,
        "is_correct":           is_correct,
        "confidence":           confidence,
        "tp":                   tp,
        "fp":                   fp,
        "fn":                   fn,
        "total_gt_fields":      total_gt_fields,
        "recovered_attributes": recovered,
        "evidence_used":        parsed.get("evidence_used", []),
        "cross_evidence_links": parsed.get("cross_evidence_links", []),
        "short_explanation":    parsed.get("short_explanation", ""),
        "elapsed_s":            elapsed,
        "raw_response":         raw,
    }


print("Evaluation runner defined.")

---
## Section 7 — Quick Test

Run **one case (case_001 / person_1), raw mode only, 3 images** to verify the pipeline end-to-end before committing to the full evaluation.

In [ ]:
print("=" * 70)
print("  QUICK TEST — case_001, raw mode, 3 images")
print("=" * 70)

qt_result = run_case(CASES[0], mode="raw", quick_test=True)

print(f"\n  Predicted : {qt_result['predicted_candidate_id']}")
print(f"  True      : {qt_result['true_candidate_id']}")
print(f"  Correct   : {qt_result['is_correct']}")
print(f"  Confidence: {qt_result['confidence']:.2f}")
print(f"  TP/FP/FN  : {qt_result['tp']}/{qt_result['fp']}/{qt_result['fn']}")
print(f"  Time      : {qt_result['elapsed_s']}s")
print(f"\n  Explanation: {qt_result['short_explanation']}")
print(f"\n  Recovered attributes:")
for k, v in qt_result['recovered_attributes'].items():
    if v is not None:
        print(f"    {k:25s}: {v}")

print("\n" + "─" * 70)
print("  Raw LLM response:")
print(qt_result['raw_response'])

---
## Section 8 — Full Evaluation

Set `QUICK_TEST = False` in Section 0 to run all 3 cases × 3 modes.

In [ ]:
all_results: list[dict] = []

if QUICK_TEST:
    print("QUICK_TEST=True — skipping full evaluation.")
    print("Set QUICK_TEST=False in Section 0 and re-run from Section 3 onwards.")
else:
    modes = EVAL_MODES
    print(f"Full evaluation: {len(CASES)} cases × {len(modes)} modes = {len(CASES)*len(modes)} total runs")
    print("=" * 72)

    for case in CASES:
        for mode in modes:
            print(f"\n→ {case['case_id']} | {mode}")
            result = run_case(case, mode, quick_test=False)
            all_results.append(result)
            status = "✅" if result["is_correct"] else "❌"
            print(f"  {status} predicted={result['predicted_candidate_id']}  "
                  f"conf={result['confidence']:.2f}  "
                  f"TP={result['tp']} FP={result['fp']} FN={result['fn']}  "
                  f"({result['elapsed_s']}s)")
            time.sleep(1)  # polite rate-limiting

    print("\n" + "=" * 72)
    print(f"  Evaluation complete — {len(all_results)} results collected.")

---
## Section 9 — Metrics

In [ ]:
def compute_mode_metrics(results: list[dict]) -> pd.DataFrame:
    """Aggregate per-mode metrics from the full result list."""
    rows = []
    modes_seen = []
    for mode in EVAL_MODES:
        mode_results = [r for r in results if r["mode"] == mode]
        if not mode_results:
            continue
        modes_seen.append(mode)

        n          = len(mode_results)
        n_correct  = sum(r["is_correct"] for r in mode_results)
        tp_total   = sum(r["tp"] for r in mode_results)
        fp_total   = sum(r["fp"] for r in mode_results)
        fn_total   = sum(r["fn"] for r in mode_results)
        gt_total   = sum(r["total_gt_fields"] for r in mode_results)
        avg_conf   = np.mean([r["confidence"] for r in mode_results])

        identity_acc = n_correct / n if n else 0.0
        attr_rec_rate = tp_total / gt_total if gt_total else 0.0
        precision    = tp_total / (tp_total + fp_total) if (tp_total + fp_total) else 0.0
        recall       = tp_total / (tp_total + fn_total) if (tp_total + fn_total) else 0.0
        f1           = (2 * precision * recall / (precision + recall)
                        if (precision + recall) else 0.0)

        rows.append({
            "mode":                   mode,
            "identity_accuracy":      round(identity_acc, 3),
            "attribute_recovery_rate": round(attr_rec_rate, 3),
            "precision":              round(precision, 3),
            "recall":                 round(recall, 3),
            "f1":                     round(f1, 3),
            "avg_confidence":         round(float(avg_conf), 3),
            "n_cases":                n,
        })

    return pd.DataFrame(rows).set_index("mode")


def per_case_table(results: list[dict]) -> pd.DataFrame:
    """Per-case result table for qualitative inspection."""
    rows = []
    for r in results:
        rows.append({
            "case_id":       r["case_id"],
            "mode":          r["mode"],
            "true_id":       r["true_candidate_id"],
            "predicted_id":  r["predicted_candidate_id"],
            "correct":       r["is_correct"],
            "tp/fp/fn":      f"{r['tp']}/{r['fp']}/{r['fn']}",
            "confidence":    r["confidence"],
        })
    return pd.DataFrame(rows)


if all_results:
    metrics_df  = compute_mode_metrics(all_results)
    per_case_df = per_case_table(all_results)

    print("\n=== SUMMARY METRICS BY MODE ===")
    print(metrics_df.to_string())
    print("\n=== PER-CASE RESULTS ===")
    print(per_case_df.to_string(index=False))
else:
    print("No full-evaluation results yet (QUICK_TEST=True). Run Section 8 with QUICK_TEST=False.")

---
## Section 10 — Visualizations

In [ ]:
def plot_evaluation_summary(metrics_df: pd.DataFrame, save_dir: Path | None = None):
    """Bar charts for key metrics across modes."""
    mode_labels = list(metrics_df.index)
    colors = ["#e74c3c", "#f39c12", "#27ae60"][:len(mode_labels)]
    x = np.arange(len(mode_labels))

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    fig.suptitle("Privacy Leakage Evaluation — Preprocessing Mode Comparison",
                 fontsize=14, fontweight="bold", y=1.02)

    def _bar(ax, values, title, ylabel, ylim=(0, 1.05)):
        bars = ax.bar(x, values, color=colors, edgecolor="white", linewidth=1.2, width=0.5)
        ax.set_xticks(x)
        ax.set_xticklabels(mode_labels, fontsize=11)
        ax.set_title(title, fontsize=12, fontweight="bold", pad=10)
        ax.set_ylabel(ylabel, fontsize=10)
        ax.set_ylim(*ylim)
        ax.spines[["top", "right"]].set_visible(False)
        for bar, val in zip(bars, values):
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02,
                    f"{val:.2f}", ha="center", va="bottom", fontsize=11, fontweight="bold")

    _bar(axes[0], metrics_df["identity_accuracy"].values,
         "Identity Matching Accuracy", "Accuracy")
    _bar(axes[1], metrics_df["attribute_recovery_rate"].values,
         "Sensitive Attribute Recovery Rate", "Recovery Rate")
    _bar(axes[2], metrics_df["avg_confidence"].values,
         "Average Agent Confidence", "Confidence")

    plt.tight_layout()
    if save_dir:
        path = save_dir / "evaluation_summary.png"
        fig.savefig(str(path), dpi=200, bbox_inches="tight")
        print(f"  Saved: {path}")
    plt.show()


def plot_prf_comparison(metrics_df: pd.DataFrame, save_dir: Path | None = None):
    """Grouped bar chart: Precision / Recall / F1 by mode."""
    mode_labels = list(metrics_df.index)
    x = np.arange(len(mode_labels))
    w = 0.22

    fig, ax = plt.subplots(figsize=(9, 5))
    ax.bar(x - w, metrics_df["precision"], w, label="Precision", color="#3498db", alpha=0.85)
    ax.bar(x,     metrics_df["recall"],    w, label="Recall",    color="#e74c3c", alpha=0.85)
    ax.bar(x + w, metrics_df["f1"],        w, label="F1",        color="#2ecc71", alpha=0.85)

    ax.set_xticks(x)
    ax.set_xticklabels(mode_labels, fontsize=11)
    ax.set_ylim(0, 1.15)
    ax.set_title("Attribute Recovery: Precision / Recall / F1 by Mode",
                 fontsize=12, fontweight="bold", pad=10)
    ax.set_ylabel("Score", fontsize=10)
    ax.legend(fontsize=10)
    ax.spines[["top", "right"]].set_visible(False)
    plt.tight_layout()

    if save_dir:
        path = save_dir / "prf_comparison.png"
        fig.savefig(str(path), dpi=200, bbox_inches="tight")
        print(f"  Saved: {path}")
    plt.show()


if all_results:
    plot_evaluation_summary(metrics_df, save_dir=OUTPUTS_DIR)
    plot_prf_comparison(metrics_df, save_dir=OUTPUTS_DIR)
else:
    print("No results to plot yet.")

---
## Section 11 — Qualitative Inspection

Side-by-side agent explanations for each case, across all modes.

In [ ]:
if all_results:
    for case in CASES:
        cid = case["case_id"]
        print(f"\n{'═'*72}")
        print(f"  CASE: {cid}  |  True answer: {case['true_candidate_id']}")
        print(f"{'═'*72}")

        for mode in EVAL_MODES:
            r = next((x for x in all_results
                      if x["case_id"] == cid and x["mode"] == mode), None)
            if r is None:
                continue
            correct_sym = "✅" if r["is_correct"] else "❌"
            print(f"\n  [{mode.upper():>14}]  {correct_sym}  predicted={r['predicted_candidate_id']}  "
                  f"conf={r['confidence']:.2f}  TP={r['tp']} FP={r['fp']} FN={r['fn']}")
            print(f"  Explanation : {r['short_explanation']}")

            recovered = {k: v for k, v in r["recovered_attributes"].items() if v is not None}
            if recovered:
                print(f"  Recovered   :")
                for k, v in recovered.items():
                    print(f"    {k:25s}: {v}")
            else:
                print(f"  Recovered   : (none)")

    print(f"\n{'═'*72}")
    print("Qualitative review complete.")
else:
    print("No full-evaluation results yet.")

---
## Section 12 — Save Results

In [ ]:
if all_results:
    # Full JSON dump (includes raw LLM responses)
    results_path = OUTPUTS_DIR / "evaluation_results.json"
    with open(results_path, "w") as f:
        json.dump(all_results, f, indent=2, default=str)
    print(f"Full results saved → {results_path}")

    # CSV summaries
    metrics_df.to_csv(OUTPUTS_DIR / "metrics_by_mode.csv")
    per_case_df.to_csv(OUTPUTS_DIR / "per_case_results.csv", index=False)
    print(f"Metric CSVs saved → {OUTPUTS_DIR}")
else:
    # Save quick-test result
    qt_path = OUTPUTS_DIR / "quick_test_result.json"
    with open(qt_path, "w") as f:
        json.dump(qt_result, f, indent=2, default=str)
    print(f"Quick-test result saved → {qt_path}")